### RAG PIPELINE - data ingestion to VECTOR DB pipeline

In [15]:
import os
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory """
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            ## Add souces information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f" Load {len(documents)} pages")
        except Exception as e:
            print(f" Error: {e}")
    
    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

# process all PDFs in the data directory

all_pdf_documents = process_all_pdfs("../data/pdf")
        

found 2 PDF files to process

 Processing: Agentic_AI_in_Education_State_of_the_Art_and_Future_Directions.pdf
 Load 25 pages

 Processing: GeaAi.pdf
 Load 9 pages

 Total documents loaded: 34


In [16]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.24; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-10-15T14:00:35+05:30', 'source': '..\\data\\pdf\\Agentic_AI_in_Education_State_of_the_Art_and_Future_Directions.pdf', 'file_path': '..\\data\\pdf\\Agentic_AI_in_Education_State_of_the_Art_and_Future_Directions.pdf', 'total_pages': 25, 'format': 'PDF 1.4', 'title': 'Agentic AI in Education: State of the Art and Future Directions', 'author': '', 'subject': 'IEEE Access;2025;13; ;10.1109/ACCESS.2025.3620473', 'keywords': '', 'moddate': '2025-10-15T22:24:17-04:00', 'trapped': 'False', 'modDate': "D:20251015222417-04'00'", 'creationDate': "D:20251015140035+05'30'", 'page': 0, 'source_file': 'Agentic_AI_in_Education_State_of_the_Art_and_Future_Directions.pdf', 'file_type': 'pdf'}, page_content='IEEE EDUCATION SOCIETY SECTION\nReceived 24 September 2025, accepted 6 October 2025, date of publication 13 October 2025,

In [17]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performace """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    ## Show example of chunk
    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]} ...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [18]:
chunks = split_documents(all_pdf_documents)

Split 34 documents into 193 chunks

 Example chunk:
Content: IEEE EDUCATION SOCIETY SECTION
Received 24 September 2025, accepted 6 October 2025, date of publication 13 October 2025, date of current version 17 October 2025.
Digital Object Identifier 10.1109/ACCE ...
Metadata: {'producer': 'pdfTeX-1.40.24; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-10-15T14:00:35+05:30', 'source': '..\\data\\pdf\\Agentic_AI_in_Education_State_of_the_Art_and_Future_Directions.pdf', 'file_path': '..\\data\\pdf\\Agentic_AI_in_Education_State_of_the_Art_and_Future_Directions.pdf', 'total_pages': 25, 'format': 'PDF 1.4', 'title': 'Agentic AI in Education: State of the Art and Future Directions', 'author': '', 'subject': 'IEEE Access;2025;13; ;10.1109/ACCESS.2025.3620473', 'keywords': '', 'moddate': '2025-10-15T22:24:17-04:00', 'trapped': 'False', 'modDate': "D:20251015222417-04'00'", 'creationDate': "D:20251015140035+05'3

### Embedding And VectorStore DB 

In [19]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [20]:
class EmbdeddingManager:
    """Handles document embedding generation using Sentence Transformer"""
    
    def __init__(self,model_name:str="all-miniLM-L6-v2"):
        """ 
        Initalize the embedding manager 
        
        Args:
            model_name: HuggingFace Model name for setence embeddings 
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """ Load the sentencetransformer model """
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise
    def generate_embeddings(self,texts:List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(text), embedding_dim)
        """
        
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embedding for {len(texts)} ...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
### initalize the embedding manager
embdedding_Manager = EmbdeddingManager()
embdedding_Manager

Loading embedding model:all-miniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6344.09it/s]


Model loaded successfully. Embedding dimension: 384


### Vector store

In [21]:
class VectorStore:
    """Manages document embeddings in ChromaDB vector store """
    
    def __init__(self, collection_name:str = "pdf_documents",persist_directory:str = "../data/vector_store"):
        """
        Initialize the vector store 

        Args:
            collection_name : Name of the chromaDB collection ".
            persist_directory: Directory to persist the vector store".
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initalize ChromaDb client and collection """
        try:
            # create persistance chromaDB client 
            self.client = chromadb.PersistentClient(path=self.persist_directory) 
            
            # Get or create collection 
            self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF document embeddings for RAG"})

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore =VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 193


In [22]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.24; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-10-15T14:00:35+05:30', 'source': '..\\data\\pdf\\Agentic_AI_in_Education_State_of_the_Art_and_Future_Directions.pdf', 'file_path': '..\\data\\pdf\\Agentic_AI_in_Education_State_of_the_Art_and_Future_Directions.pdf', 'total_pages': 25, 'format': 'PDF 1.4', 'title': 'Agentic AI in Education: State of the Art and Future Directions', 'author': '', 'subject': 'IEEE Access;2025;13; ;10.1109/ACCESS.2025.3620473', 'keywords': '', 'moddate': '2025-10-15T22:24:17-04:00', 'trapped': 'False', 'modDate': "D:20251015222417-04'00'", 'creationDate': "D:20251015140035+05'30'", 'page': 0, 'source_file': 'Agentic_AI_in_Education_State_of_the_Art_and_Future_Directions.pdf', 'file_type': 'pdf'}, page_content='IEEE EDUCATION SOCIETY SECTION\nReceived 24 September 2025, accepted 6 October 2025, date of publication 13 October 2025,

In [23]:
### convert the text to embeddings
texts = [doc.page_content for doc in chunks]

## we will generate the embeddings
embeddings = embdedding_Manager.generate_embeddings(texts)

## store in the vector database

vectorstore.add_documents(chunks,embeddings)




Generating embedding for 193 ...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches: 100%|██████████| 7/7 [00:11<00:00,  1.66s/it]


Generated embeddings with shape: (193, 384)
Adding 193 documents to vector store...
Successfully added 193 documents to vector store
Total documents in collection: 386
